In [1]:
import os
import pickle
import string
from stanfordcorenlp import StanfordCoreNLP
from nltk.corpus import stopwords
from tqdm import tqdm


In [2]:
nlp = StanfordCoreNLP('../../stanford-corenlp-4.5.6', lang='en')

In [3]:
#路径设置
os.path.abspath(os.path.dirname(os.path.dirname('__file__')))
os.path.abspath(os.path.dirname(os.getcwd()))
os.path.abspath(os.path.join(os.getcwd(), ".."))
##修改文件名qprop_small为qprop
dataset ='new2'

In [4]:
input = os.sep.join(['..','..', 'data_tgcn', dataset, dataset+"_new"])
output = os.sep.join(['..','..', 'data_tgcn', dataset, 'stanford'])
if not os.path.exists(output):
    os.mkdir(output)

In [5]:
#读取病例
yic_content_list = []
f = open(input + '.clean.txt', 'r', encoding="gbk")
lines = f.readlines()

In [6]:
for line in lines:
    yic_content_list.append(line.strip())#文本条
f.close()

In [7]:
yic_content_list2 = yic_content_list[:5000]

# 打印新列表，验证是否正确包含了前十个元素
#for doc in yic_content_list2:
    #print(doc)

In [8]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
stop_words = set(stopwords.words('english'))

In [10]:
def calculate_distance(dependencies, num_words):
    distances = [[float('inf')] * num_words for _ in range(num_words)]
    for i in range(num_words):
        distances[i][i] = 0  # 自距离为0
    
    for dep in dependencies:
        rel, gov, dep = dep
        # 确保索引在有效范围内
        if 0 < gov <= num_words and 0 < dep <= num_words:
            gov_index = gov - 1
            dep_index = dep - 1
            distances[gov_index][dep_index] = 1
            distances[dep_index][gov_index] = 1  # 双向，如果需要无向距离

    # 使用Floyd-Warshall算法计算所有对最短路径
    for k in range(num_words):
        for i in range(num_words):
            for j in range(num_words):
                if distances[i][j] > distances[i][k] + distances[k][j]:
                    distances[i][j] = distances[i][k] + distances[k][j]
    return distances


In [11]:
def create_mask_matrix(distances, threshold=3):
    num_words = len(distances)
    mask_matrix = [[0 if distances[i][j] > threshold else 1 for j in range(num_words)] for i in range(num_words)]
    return mask_matrix


In [12]:
#获取句法依存关系对+加入句法掩码矩阵

rela_pair_count_str = {}
for doc_id in tqdm(range(0,len(yic_content_list))): #使用 tqdm 进度条包装器遍历文档列表，提供一个用户友好的进度条。
# for doc_id in range(len(yic_content_list)):
#     print(doc_id)
    words = yic_content_list[doc_id]
    words = words.split("\n")
    rela=[]
    for window in words:
        if window==' ':
            continue
        #构造rela_pair_count
        window = window.replace(string.punctuation, ' ')#把所有的标点符号替换为' '
        ws=window.split(" ")
        res = nlp.dependency_parse(window) #生成句法依存对   res 包含依存关系的类型和涉及的词索引。
        
        for tuple in res:
            # rela.append(str(tuple[0]) + ', ' + str(tuple[1]))
            rela.append(str(tuple[0]) + ', ' + str(tuple[1])+', '+str(tuple[2])) #rela.append(...): 将解析得到的依存关系（类型和词索引）格式化为字符串并添加到列表 rela 中。
            
        for pair in rela:
            #print(pair)
            pair=pair.split(", ")
            rel_type = pair[0]  # 关系类型（未用于计算，但可用于其他目的）
            gov_index = int(pair[1])  # 主控词索引
            dep_index = int(pair[2])  # 从属词索引

            # 计算索引的绝对差值
            index_difference = abs(gov_index - dep_index)
            if pair[0]=='ROOT' or pair[1]=='ROOT':
                continue
            # if pair[0] == pair[1]:
            if pair[2] == pair[1]:
                continue
            if int(pair[1])>len(ws) or int(pair[2])>len(ws):
                continue
            # if pair[0] in string.punctuation or pair[1] in string.punctuation:
            if ws[int(pair[2])-1] in string.punctuation or ws[int(pair[1])-1] in string.punctuation:
                continue
            # if pair[0] in stop_words or pair[1] in stop_words:
            if ws[int(pair[1])-1] in stop_words or ws[int(pair[2])-1] in stop_words:
                continue
            # word_pair_str = pair[0]+ ',' + pair[1]
            word_pair_str = ws[int(pair[1])-1] + ',' + ws[int(pair[2])-1]
            if word_pair_str in rela_pair_count_str and index_difference < 4 :
                rela_pair_count_str[word_pair_str] += 1
            else:
                if word_pair_str not in rela_pair_count_str and index_difference < 4:
                    rela_pair_count_str[word_pair_str] = 1  # 将该键的值设为1
            # two orders
            # word_pair_str = pair[1]+ ',' + pair[0]
            # print(int(pair[2])-1,int(pair[1])-1)
            word_pair_str = ws[int(pair[2])-1] + ',' + ws[int(pair[1])-1]
            if word_pair_str in rela_pair_count_str and index_difference < 4:
                rela_pair_count_str[word_pair_str] += 1
            else:
                if word_pair_str not in rela_pair_count_str and index_difference < 4:
                    rela_pair_count_str[word_pair_str] = 1  # 将该键的值设为1
            '''
            #print(index_difference)
            #print(pair)
            #print(word_pair_str)   #word_pair_str包含的内容为单词对
            if word_pair_str in rela_pair_count_str and index_difference < 4:
                rela_pair_count_str[word_pair_str] += 1
            else:
                rela_pair_count_str[word_pair_str] = 1
            # two orders
            # word_pair_str = pair[1]+ ',' + pair[0]
            # print(int(pair[2])-1,int(pair[1])-1)
            word_pair_str = ws[int(pair[2])-1] + ',' + ws[int(pair[1])-1]
            if word_pair_str in rela_pair_count_str and index_difference < 4:
                rela_pair_count_str[word_pair_str] += 1
            else:
                rela_pair_count_str[word_pair_str] = 1
            #print(rela_pair_count_str[word_pair_str])
            
print(rela_pair_count_str)
'''

100%|██████████| 19263/19263 [11:31<00:00, 27.86it/s]


In [13]:
#将rela_pair_count_str存成pkl格式
output1=open(output + '/{}_stan.pkl'.format(dataset),'wb')
pickle.dump(rela_pair_count_str, output1)